In [29]:
import os
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

## 负荷权重

In [4]:
root_path = 'raw_data/'
data_path = os.listdir(root_path)
COLUMNS = ['Date', 'COAST', 'EAST', 'FWEST', 'NORTH', 'NCENT', 'SOUTH', 'SCENT', 'WEST', 'ERCOT']

# 数据合并
df_all = []
for i, v in enumerate(data_path):
    df = pd.read_excel(os.path.join(root_path, v))
    df.columns = COLUMNS
    df_all.append(df)
    
df = pd.concat(df_all)
df = df.fillna(method='ffill')

In [9]:
columns = df.columns[1:-1]
weights = []
for v in columns:
    weights.append(df[v].sum() / df.ERCOT.sum())

In [26]:
weight_ser = pd.Series(weights)
weight_ser.index = columns
weight_ser = weight_ser.sort_values(ascending=False)
weight_ser

NCENT    0.306864
COAST    0.281928
SCENT    0.161343
FWEST    0.084180
SOUTH    0.081722
EAST     0.034900
WEST     0.027553
NORTH    0.021495
dtype: float64

## 天气数据

### 实际天气2016-2017（因预测天气从2018年开始）

In [19]:
"""
ERCOT负荷加权天气实际数据下载
==============================
使用Open-Meteo Historical Weather API (ERA5再分析数据)
数据范围: 1940年至今
"""

import requests
import pandas as pd
import numpy as np

# ERCOT各Zone坐标和负荷权重
ERCOT_ZONES = {
    'Coast':        {'lat': 29.76, 'lon': -95.37, 'weight': 0.28}, 
    'North_Central':{'lat': 32.78, 'lon': -96.80, 'weight': 0.31}, 
    'South_Central':{'lat': 29.42, 'lon': -98.49, 'weight': 0.16},  
    'West':         {'lat': 31.99, 'lon': -102.08, 'weight': 0.03},
    'South':        {'lat': 26.20, 'lon': -98.23, 'weight': 0.08},
    'North':        {'lat': 33.91, 'lon': -98.49, 'weight': 0.02},
    'East':         {'lat': 32.35, 'lon': -94.71, 'weight': 0.04},
    'Far_West':     {'lat': 31.76, 'lon': -106.49, 'weight': 0.08},
}

def get_ercot_weighted_actual(start_date, end_date):
    """获取ERCOT负荷加权的实际天气数据"""
    
    # 改用Historical Weather API (实际数据)
    url = "https://archive-api.open-meteo.com/v1/archive"
    all_zones = []
    
    for zone, info in ERCOT_ZONES.items():
        params = {
            "latitude": info['lat'],
            "longitude": info['lon'],
            "start_date": start_date,
            "end_date": end_date,
            "hourly": [
                # 核心温度
                "temperature_2m",
                "apparent_temperature",
                "relative_humidity_2m",

                # 极端天气
                "precipitation",
                "rain",
                "snowfall",
                "weather_code",
            ],
            "timezone": "America/Chicago"
            # 注意: 实际数据API不需要 "models" 参数
        }
        
        response = requests.get(url, params=params)
        if response.status_code == 200:
            data = response.json()
            df = pd.DataFrame(data['hourly'])
            df['zone'] = zone
            df['weight'] = info['weight']
            all_zones.append(df)
        
        print(f"{zone} 完成提取！")
    
    # 合并数据
    combined = pd.concat(all_zones, ignore_index=True)
    combined['time'] = pd.to_datetime(combined['time'])
    
    # 计算加权平均
    weighted_avg = combined.groupby('time').apply(
        lambda x: pd.Series({
            'temperature_2m': np.average(x['temperature_2m'], weights=x['weight']),
            'relative_humidity_2m': np.average(x['relative_humidity_2m'], weights=x['weight']),
            'apparent_temperature': np.average(x['apparent_temperature'], weights=x['weight']),
            
            'precipitation': np.average(x['precipitation'], weights=x['weight']),
            'rain': np.average(x['rain'], weights=x['weight']),
            'snowfall': np.average(x['snowfall'], weights=x['weight']),
            'weather_code': np.average(x['weather_code'], weights=x['weight']),
        })
    ).reset_index()
    
    return weighted_avg, combined

# 使用示例
weighted_actual, zone_details = get_ercot_weighted_actual("2016-01-01", "2017-12-31")

Coast 完成提取！
North_Central 完成提取！
South_Central 完成提取！
West 完成提取！
South 完成提取！
North 完成提取！
East 完成提取！
Far_West 完成提取！


In [21]:
weighted_actual.shape[0] / 24

731.0

In [23]:
# 保存
weighted_actual.to_csv('./weather_data/weighted_actual.csv', index=False)

In [30]:
import requests
import pandas as pd
import numpy as np

# ERCOT各Zone坐标和负荷权重（基于历史负荷占比估算）
ERCOT_ZONES = {
    'Coast':        {'lat': 29.76, 'lon': -95.37, 'weight': 0.28}, 
    'North_Central':{'lat': 32.78, 'lon': -96.80, 'weight': 0.31}, 
    'South_Central':{'lat': 29.42, 'lon': -98.49, 'weight': 0.16},  
    'West':         {'lat': 31.99, 'lon': -102.08, 'weight': 0.03},
    'South':        {'lat': 26.20, 'lon': -98.23, 'weight': 0.08},
    'North':        {'lat': 33.91, 'lon': -98.49, 'weight': 0.02},
    'East':         {'lat': 32.35, 'lon': -94.71, 'weight': 0.04},
    'Far_West':     {'lat': 31.76, 'lon': -106.49, 'weight': 0.08},
}

def get_ercot_weighted_forecast(start_date, end_date):
    """获取ERCOT负荷加权的天气预报"""
    
    url = "https://historical-forecast-api.open-meteo.com/v1/forecast"
    all_zones = []
    
    for zone, info in ERCOT_ZONES.items():
        params = {
            "latitude": info['lat'],
            "longitude": info['lon'],
            "start_date": start_date,
            "end_date": end_date,
            "hourly": [
                # 核心温度
                "temperature_2m",
                "apparent_temperature",
                "relative_humidity_2m",

                # 极端天气
                "precipitation",
                "rain",
                "snowfall",
                "weather_code",
            ],
            "timezone": "America/Chicago",
            "models": "gfs_seamless"
        }
        
        response = requests.get(url, params=params)
        if response.status_code == 200:
            data = response.json()
            df = pd.DataFrame(data['hourly'])
            df['zone'] = zone
            df['weight'] = info['weight']
            all_zones.append(df)
        
        print(f"{zone} 完成提取！")
    
    # 合并数据
    combined = pd.concat(all_zones, ignore_index=True)
    combined['time'] = pd.to_datetime(combined['time'])
    
    # 计算加权平均
    weighted_avg = combined.groupby('time').apply(
        lambda x: pd.Series({
            'temperature_2m': np.average(x['temperature_2m'], weights=x['weight']),
            'relative_humidity_2m': np.average(x['relative_humidity_2m'], weights=x['weight']),
            'apparent_temperature': np.average(x['apparent_temperature'], weights=x['weight']),
            
            'precipitation': np.average(x['precipitation'], weights=x['weight']),
            'rain': np.average(x['rain'], weights=x['weight']),
            'snowfall': np.average(x['snowfall'], weights=x['weight']),
            'weather_code': np.average(x['weather_code'], weights=x['weight']),
        })
    ).reset_index()
    
    return weighted_avg, combined

# 使用示例
weighted_forecast, zone_details = get_ercot_weighted_forecast("2018-01-01", "2024-12-31")

Coast 完成提取！
North_Central 完成提取！
South_Central 完成提取！
West 完成提取！
South 完成提取！
North 完成提取！
East 完成提取！
Far_West 完成提取！


In [31]:
weighted_forecast.shape[0] / 24

2557.0

In [32]:
# 保存
weighted_forecast.to_csv('./weather_data/weighted_forecast.csv', index=False)